<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2024/2025  

<img src="../../img/portada.png" width="300"/>

---

#  Ensamble Secuencial de Modelos Predictivos  
### Primera Convocatoria – Junio 2025  

**Profesor:** Juan Galán Páez  
**Autores:**  
Carlos Martín de Prado Barragán  
Marco Padilla Gómez  

**Fecha de entrega:** 9 de Junio de 2025  

</div>

## Introducción

En este notebook se realiza una exploración exhaustiva de los hiperparámetros del meta-modelo secuencial aplicado al conjunto de datos sobre la enfermedad de Parkinson.

Mediante validación cruzada con 5 particiones, se evalúan diferentes combinaciones de parámetros clave como el número de estimadores, la tasa de aprendizaje y la profundidad máxima de los árboles base.

Además del coeficiente de determinación R², se incluyen métricas complementarias como el Error Absoluto Medio (MAE) y la Raíz del Error Cuadrático Medio (RMSE) para proporcionar una evaluación más completa del rendimiento.

El objetivo es identificar las configuraciones que maximicen la capacidad predictiva y la robustez del modelo, fundamentando así las decisiones de diseño y optimización.

## 1. Importación de librerías y configuración del entorno

- Se añade la ruta `../../src` para importar el módulo personalizado `sequential_ensemble.py` que contiene la implementación del meta-modelo secuencial.
- Se importan las librerías necesarias para el procesamiento de datos (`numpy`, `pandas`), modelado (`DecisionTreeRegressor`), preprocesamiento (`StandardScaler`), validación cruzada (`cross_validate`, `KFold`), y métricas de evaluación (`mean_absolute_error`, `mean_squared_error`).
- Finalmente, se importa la clase `SequentialEnsembleRegressor` para utilizar el meta-modelo en los experimentos.

In [1]:
# Importación de librerías y configuración del entorno
import sys
sys.path.append('../../src')

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sequential_ensemble import SequentialEnsembleRegressor

## 2. Carga y preprocesamiento de datos

- Se carga el conjunto de datos `parkinsons.csv` desde la carpeta `data`.
- Se separan las variables predictoras (`X`) de la variable objetivo (`y`), que en este caso es la columna `total_UPDRS`.
- Se aplica estandarización a las variables predictoras utilizando `StandardScaler`, para normalizar su escala y facilitar el entrenamiento del modelo.


In [2]:
# Carga y preprocesamiento de datos
df = pd.read_csv("../../data/parkinsons.csv")
X = df.drop(columns=["total_UPDRS"])
y = df["total_UPDRS"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 3. Definición del espacio de búsqueda de hiperparámetros

- Se establecen los rangos de valores para los hiperparámetros clave a explorar:
  - `n_estimators`: número de modelos base en el ensamblador, con valores 10, 30 y 50.
  - `learning_rate`: tasa de aprendizaje que modula la contribución de cada modelo, con valores 0.05, 0.1 y 0.2.
  - `max_depth`: profundidad máxima permitida para los árboles de decisión, con valores 3, 4 y 5.
  - `sample_size`: porcentaje del conjunto de entrenamiento en cada iteración.
- Se inicializa una lista vacía `results` para almacenar los resultados de cada combinación probada durante la experimentación.

In [3]:
# Definimos rangos para los hiperparámetros más relevantes
n_estimators_options = [10, 30, 50]
learning_rate_options = [0.05, 0.1, 0.2]
max_depth_options = [3, 4, 5]
sample_size_options = [0.6, 0.75, 0.9]  

# Para guardar resultados
results = []

## 4. Ejecución de experimentos con validación cruzada

- Se configura la validación cruzada con 5 particiones (`KFold`), mezclando aleatoriamente los datos para mayor representatividad.
- Se realiza un bucle anidado para probar todas las combinaciones posibles de los hiperparámetros definidos: número de estimadores, tasa de aprendizaje, profundidad máxima y porcentaje del conjunto de entrenamiento.
- Para cada configuración, se crea una instancia del meta-modelo secuencial con esos parámetros.
- Se ejecuta la validación cruzada calculando múltiples métricas:  
  - Coeficiente de determinación R²,  
  - Error Absoluto Medio (MAE),  
  - Raíz del Error Cuadrático Medio (RMSE).
- Los resultados (medias y desviaciones estándar) se almacenan en la lista `results` para su posterior análisis.
- Se imprime un resumen en cada iteración para seguimiento en tiempo real de los experimentos.

In [4]:
# Configurar validación cruzada
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Métricas a evaluar como diccionario para evitar NaN
scoring = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error'
}

# Bucle para probar combinaciones con validación cruzada
for sample_size in sample_size_options:
    for n_estimators in n_estimators_options:
        for lr in learning_rate_options:
            for max_depth in max_depth_options:
                modelo = SequentialEnsembleRegressor(
                    base_estimator=DecisionTreeRegressor,
                    n_estimators=n_estimators,
                    lr=lr,
                    sample_size=sample_size,
                    est_params={"max_depth": max_depth},
                    random_state=42
                )
                scores = cross_validate(modelo, X_scaled, y, cv=kf, scoring=scoring)

                mean_r2 = np.mean(scores['test_r2'])
                mean_mae = -np.mean(scores['test_mae'])
                mean_rmse = np.sqrt(-np.mean(scores['test_mse']))
                std_r2 = np.std(scores['test_r2'])

                results.append({
                    "sample_size": sample_size,
                    "n_estimators": n_estimators,
                    "learning_rate": lr,
                    "max_depth": max_depth,
                    "mean_r2": mean_r2,
                    "std_r2": std_r2,
                    "mean_mae": mean_mae,
                    "mean_rmse": mean_rmse
                })

                print(f"Evaluado sample_size={sample_size}, n_estimators={n_estimators}, "
                      f"lr={lr}, max_depth={max_depth} -> R2 mean: {mean_r2:.4f}")

Evaluado sample_size=0.6, n_estimators=10, lr=0.05, max_depth=3 -> R2 mean: -2.3694
Evaluado sample_size=0.6, n_estimators=10, lr=0.05, max_depth=4 -> R2 mean: -2.2993
Evaluado sample_size=0.6, n_estimators=10, lr=0.05, max_depth=5 -> R2 mean: -2.2352
Evaluado sample_size=0.6, n_estimators=10, lr=0.1, max_depth=3 -> R2 mean: -0.4977
Evaluado sample_size=0.6, n_estimators=10, lr=0.1, max_depth=4 -> R2 mean: -0.4044
Evaluado sample_size=0.6, n_estimators=10, lr=0.1, max_depth=5 -> R2 mean: -0.3115
Evaluado sample_size=0.6, n_estimators=10, lr=0.2, max_depth=3 -> R2 mean: 0.4283
Evaluado sample_size=0.6, n_estimators=10, lr=0.2, max_depth=4 -> R2 mean: 0.5689
Evaluado sample_size=0.6, n_estimators=10, lr=0.2, max_depth=5 -> R2 mean: 0.6525
Evaluado sample_size=0.6, n_estimators=30, lr=0.05, max_depth=3 -> R2 mean: 0.1228
Evaluado sample_size=0.6, n_estimators=30, lr=0.05, max_depth=4 -> R2 mean: 0.2473
Evaluado sample_size=0.6, n_estimators=30, lr=0.05, max_depth=5 -> R2 mean: 0.3412
Eval

## 5. Análisis y visualización de resultados

- Se convierte la lista de resultados en un DataFrame de Pandas para facilitar su manipulación.
- Se ordenan las configuraciones probadas según el valor medio del coeficiente de determinación R², de mayor a menor, para identificar rápidamente las mejores combinaciones.
- Se reinicia el índice del DataFrame para mantener la consistencia tras el ordenamiento.
- Finalmente, se muestra la tabla con todas las configuraciones evaluadas y sus respectivos resultados, facilitando el análisis comparativo.

In [5]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="mean_r2", ascending=False)
results_df.reset_index(drop=True, inplace=True)
results_df

,sample_size,n_estimators,learning_rate,max_depth,mean_r2,std_r2,mean_mae,mean_rmse
0,0.90,50,0.20,5,0.886473,0.016948,2.601058,3.574099
1,0.75,50,0.20,5,0.879749,0.016230,2.704066,3.676620
2,0.90,30,0.20,5,0.863404,0.013830,2.923917,3.925047
3,0.60,50,0.20,5,0.862029,0.020939,2.912641,3.939051
4,0.75,30,0.20,5,0.858421,0.017708,2.973937,3.989660
...,...,...,...,...,...,...,...,...
76,0.75,10,0.05,4,-2.304303,0.166673,17.237646,19.289073
77,0.90,10,0.05,4,-2.304522,0.163727,17.222312,19.289119
78,0.60,10,0.05,3,-2.369375,0.161892,17.239355,19.478563
79,0.90,10,0.05,3,-2.370172,0.166939,17.237494,19.479922


In [6]:
# Comparación del mejor ensamblador con el modelo base individual (DecisionTreeRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

# Extraer la mejor configuración del ensamblador
mejor_config = results_df.iloc[0]
mejor_max_depth = int(mejor_config["max_depth"])

# Crear y evaluar el modelo base individual con ese max_depth
modelo_base = DecisionTreeRegressor(max_depth=mejor_max_depth, random_state=42)
r2_base = np.mean(cross_val_score(modelo_base, X_scaled, y, cv=kf, scoring='r2'))

# Mostrar comparación
print(f"R² del mejor ensamblador (validación cruzada): {mejor_config['mean_r2']:.4f}")
print(f"R² del modelo base individual (max_depth={mejor_max_depth}): {r2_base:.4f}")

R² del mejor ensamblador (validación cruzada): 0.8865
R² del modelo base individual (max_depth=5): 0.4939


## 6. Interpretación de resultados y conclusiones

Los experimentos realizados evidencian que el rendimiento del meta-modelo secuencial está fuertemente condicionado por la configuración de los hiperparámetros utilizados.

- Cuando se emplea un número reducido de estimadores (`n_estimators=10`) y una tasa de aprendizaje baja (`lr=0.05`), el modelo obtiene valores negativos de R², lo que indica que su desempeño es **peor que una predicción constante basada en la media**. Esto sugiere una capacidad insuficiente del modelo para capturar patrones significativos en los datos bajo estas condiciones.

- Al incrementar el número de estimadores y la tasa de aprendizaje, el modelo mejora notablemente su rendimiento. Por ejemplo, con `n_estimators=50`, `lr=0.2`, `max_depth=5` y `sample_size=0.9`, el modelo alcanza un R² promedio de **0.8865**, lo que significa que **explica aproximadamente el 89% de la variabilidad** de los datos de prueba. Esta es la mejor configuración encontrada.

- La **profundidad máxima (`max_depth`)** de los árboles tiene un impacto significativo: árboles más profundos permiten a cada estimador capturar relaciones más complejas, lo cual mejora el rendimiento global del meta-modelo. Sin embargo, este parámetro debe controlarse cuidadosamente para evitar el sobreajuste.

- La **tasa de aprendizaje (`lr`)** y el **número de estimadores (`n_estimators`)** presentan un equilibrio importante: una tasa baja necesita más iteraciones para corregir errores de forma efectiva, mientras que una tasa más alta puede mejorar rápidamente la predicción, pero con riesgo de sobreajuste si no se combina adecuadamente con el resto de parámetros.

- El parámetro **`sample_size`**, que determina la proporción de ejemplos usados en cada iteración, también influye en el rendimiento. En general, valores más altos (como 0.9) han proporcionado mejores resultados que valores bajos (como 0.6), probablemente debido a que permiten a cada modelo aprender de una porción más representativa del conjunto de entrenamiento.

- Las métricas complementarias (MAE y RMSE) también reflejan estas conclusiones: las configuraciones óptimas presentan errores más bajos y menor dispersión, lo que indica mayor **precisión y estabilidad** en las predicciones del ensamble.

- Finalmente, se realizó una comparación directa entre el mejor modelo ensamblador y un modelo individual `DecisionTreeRegressor` con `max_depth=5`. El ensamblador alcanzó un R² promedio de **0.8865**, mientras que el modelo base obtuvo solo **0.4939**. Esta diferencia confirma que el **aprendizaje secuencial con corrección de errores** permite construir un modelo con **mayor capacidad predictiva** que un único estimador tradicional.